# RSI(2) 전략 수작업 대조

스펙 '검증 절차 > 수작업 대조' 기록.

1. RSI(2) 전략 거래 내역에서 무작위로 5건을 뽑는다 (시드 고정)
2. 원본 가격(parquet)을 엔진 코드를 거치지 않고 직접 읽어, 진입 전일 종가 기준으로 진입 조건이 성립했는지 확인한다
3. 진입가·청산가가 해당일 시가와 일치하는지 확인한다
4. 비용 반영 수익률을 직접 계산해 엔진 값과 소수점 4자리까지 비교한다

추가로, 진입일 이후 **처음으로** 청산 조건(종가 > SMA(5) 또는 보유 10거래일)이 성립한 날의 다음 날 시가에 청산했는지도 확인한다 (늦게 또는 일찍 청산하지 않았는지).

SMA·RSI는 `swinglab.indicators`를 쓰지 않고 아래에서 파이썬 반복문으로 따로 계산한다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from swinglab.backtest import run_backtest
from swinglab.config import load_backtest_config, load_universe
from swinglab.pipeline.storage import read_prices
from swinglab.strategies import BuyAndHold, build_strategy

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
universe = load_universe(ROOT / "config")
config = load_backtest_config(ROOT / "config")
prices = read_prices(ROOT / "data/clean/prices.parquet")  # 엔진 입력
print(f"데이터: {prices['date'].min().date()} ~ {prices['date'].max().date()}, {prices['ticker'].nunique()}종목")

In [ ]:
strategy = build_strategy("rsi2_pullback", universe)
result = run_backtest(prices, strategy, config)
trades = result.trades
COST = config.cost_rate
print(f"구간 {result.start_date.date()} ~ {result.end_date.date()}, 청산 거래 {len(trades)}건, 편도 비용 {config.cost_bps:g}bp")

## 1. 무작위 5건 (시드 20261108)

In [ ]:
rng = np.random.default_rng(20261108)
sample = trades.iloc[np.sort(rng.choice(len(trades), size=5, replace=False))].reset_index(drop=True)
sample[["ticker", "entry_date", "exit_date", "entry_price", "exit_price", "return", "holding_days", "exit_reason"]]

## 2. 손계산 함수 (라이브러리와 독립)

In [ ]:
# 원본 가격을 pandas로 직접 읽는다 (swinglab 저장 코드를 거치지 않음)
raw = pd.read_parquet(ROOT / "data/clean/prices.parquet")
raw["date"] = pd.to_datetime(raw["date"])


def sma_manual(closes: list[float], n: int) -> float:
    """마지막 n개 종가의 산술평균."""
    window = closes[-n:]
    assert len(window) == n
    return sum(window) / n


def rsi_manual(closes: list[float], period: int = 2) -> float:
    """Wilder RSI를 첫 봉부터 따라 계산. 첫 평균은 단순평균, 이후 (이전 × (n−1) + 이번) / n."""
    changes = [b - a for a, b in zip(closes[:-1], closes[1:])]
    gains = [max(c, 0.0) for c in changes]
    losses = [max(-c, 0.0) for c in changes]
    avg_gain = sum(gains[:period]) / period
    avg_loss = sum(losses[:period]) / period
    for g, l in zip(gains[period:], losses[period:]):
        avg_gain = (avg_gain * (period - 1) + g) / period
        avg_loss = (avg_loss * (period - 1) + l) / period
    if avg_loss == 0:
        return 50.0 if avg_gain == 0 else 100.0
    return 100 - 100 / (1 + avg_gain / avg_loss)

## 3. 건별 대조

In [ ]:
def check(t) -> dict:
    hist = raw[raw["ticker"] == t.ticker].sort_values("date").reset_index(drop=True)
    dates, closes = list(hist["date"]), list(hist["close"])
    e, x = dates.index(t.entry_date), dates.index(t.exit_date)
    s = e - 1  # 진입 전일 = 신호일

    close_s = closes[s]
    sma200_s = sma_manual(closes[: s + 1], 200)
    rsi_s = rsi_manual(closes[: s + 1])
    entry_ok = close_s > sma200_s and rsi_s < 10

    # 진입일 종가부터 처음으로 청산 조건이 성립하는 날 (같은 날이면 신호 청산이 먼저)
    exit_signal_day, reason = None, None
    for d in range(e, len(dates)):
        if closes[d] > sma_manual(closes[: d + 1], 5):
            exit_signal_day, reason = d, "signal"
            break
        if d - e + 1 >= 10:
            exit_signal_day, reason = d, "max_hold"
            break

    entry_open, exit_open = hist.at[e, "open"], hist.at[x, "open"]
    manual_return = exit_open * (1 - COST) / (entry_open * (1 + COST)) - 1
    return {
        "ticker": t.ticker,
        "신호일": dates[s].date(),
        "신호일 종가": round(close_s, 4),
        "SMA200": round(sma200_s, 4),
        "RSI2": round(rsi_s, 2),
        "진입조건": entry_ok,
        "진입일": dates[e].date(),
        "진입가=시가": abs(t.entry_price - entry_open) < 1e-9,
        "청산신호일(수동)": dates[exit_signal_day].date(),
        "청산일 일치": exit_signal_day + 1 == x,
        "사유(수동)": reason,
        "사유 일치": reason == t.exit_reason,
        "청산가=시가": abs(t.exit_price - exit_open) < 1e-9,
        "수익률(수동)": round(manual_return, 4),
        "수익률(엔진)": round(t["return"], 4),
        "수익률 4자리 일치": round(manual_return, 4) == round(t["return"], 4),
    }


report = pd.DataFrame([check(t) for _, t in sample.iterrows()])
report

In [ ]:
checks = ["진입조건", "진입가=시가", "청산일 일치", "사유 일치", "청산가=시가", "수익률 4자리 일치"]
ok = report[checks].all(axis=None)
print("5건 모두 일치" if ok else "불일치 있음")
print(report[["ticker"] + checks].to_string(index=False))
assert ok

## 4. 원본 가격 (사람 눈으로 확인용)

건별로 신호일부터 청산일까지의 원본 시가·종가와 손으로 계산한 SMA(5).

In [ ]:
for _, t in sample.iterrows():
    hist = raw[raw["ticker"] == t.ticker].sort_values("date").reset_index(drop=True)
    closes = list(hist["close"])
    e = hist.index[hist["date"] == t.entry_date][0]
    x = hist.index[hist["date"] == t.exit_date][0]
    rows = hist.loc[e - 1 : x, ["date", "open", "close"]].copy()
    rows["SMA5"] = [sma_manual(closes[: i + 1], 5) for i in rows.index]
    rows["종가>SMA5"] = rows["close"] > rows["SMA5"]
    rows["표시"] = ""
    rows.loc[e - 1, "표시"] = "신호일"
    rows.loc[e, "표시"] = "진입(시가)"
    rows.loc[x, "표시"] = "청산(시가)"
    print(f"--- {t.ticker} {t.entry_date.date()} → {t.exit_date.date()}  엔진 수익률 {t['return']:.4%}")
    rows[["open", "close", "SMA5"]] = rows[["open", "close", "SMA5"]].round(4)
    print(rows.to_string(index=False), end="\n\n")

<!-- recorded-results -->
## 기록된 결과 (2026-11-08 실행)

코드 셀 출력은 저장소에 넣지 않으므로 대조 결과를 여기에 표로 남긴다. 노트북을 다시 실행하면 같은 표가 출력된다 (`uv run jupyter-execute --inplace notebooks/manual_check.ipynb`).

엔진: rsi2_pullback, 편도 비용 5bp, 청산 거래 1682건 중 시드 20261108로 5건 추출.

| ticker | 신호일 | RSI2 | 진입조건 | 진입일 | 진입가=시가 | 청산신호일(수동) | 청산일 일치 | 사유(수동) | 사유 일치 | 청산가=시가 | 수익률(수동) | 수익률(엔진) | 수익률 4자리 일치 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| XLE | 2011-05-04 | 5.91 | ✅ | 2011-05-05 | ✅ | 2011-05-09 | ✅ | signal | ✅ | ✅ | 0.0149 | 0.0149 | ✅ |
| XLI | 2013-04-15 | 7.53 | ✅ | 2013-04-16 | ✅ | 2013-04-23 | ✅ | signal | ✅ | ✅ | 0.0002 | 0.0002 | ✅ |
| XLU | 2016-08-04 | 9.87 | ✅ | 2016-08-05 | ✅ | 2016-08-11 | ✅ | signal | ✅ | ✅ | -0.0043 | -0.0043 | ✅ |
| XLF | 2018-06-14 | 4.08 | ✅ | 2018-06-15 | ✅ | 2018-06-28 | ✅ | max_hold | ✅ | ✅ | -0.0149 | -0.0149 | ✅ |
| XLP | 2024-06-12 | 3.15 | ✅ | 2024-06-13 | ✅ | 2024-06-17 | ✅ | signal | ✅ | ✅ | 0.0105 | 0.0105 | ✅ |

**판정: 5건 모두 일치**

## 사람 확인란

위 자동 대조는 엔진과 독립된 계산으로 5건 모두 일치를 확인한다. 스펙의 '수작업' 취지에 맞게, 아래는 사람이 4번 표의 원본 가격을 직접 보고 체크한다.

- [ ] 5건의 신호일 종가·진입일 시가·청산일 시가를 원본에서 직접 확인함
- [ ] 확인자 / 날짜: 